# Policy-Based Agents: REINFORCE on CartPole

CSCI 6353 · Topic 37.

A **policy-based** agent learns the policy $\pi_\theta(a\mid s)$ directly — feed it a state, it
outputs a probability distribution over actions — and acts by **sampling**. No value function,
no $\arg\max$. This notebook is the REINFORCE algorithm in full, trained on `CartPole-v1`.

It runs on a **CPU** (no GPU needed). On Colab: *Runtime → Run all*.

## 1. Setup

Install the environment and import everything. `Categorical` is what makes the policy
stochastic — we sample an action from it instead of taking the best one.

In [ ]:
!pip -q install gymnasium torch matplotlib

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

# Hyperparameters (as in the lecture)
learning_rate = 0.0002
gamma = 0.98

## 2. The policy network

A tiny MLP: the 4-dimensional CartPole state in, a **softmax over the 2 actions** out. The
`train_net` method is the whole of REINFORCE. It walks the finished episode **backward** to
build each return $G_t = r_t + \gamma G_{t+1}$, and accumulates the loss

$$L = -\sum_t G_t \,\log \pi_\theta(a_t\mid s_t).$$

The minus sign turns gradient **ascent** on expected return into the gradient **descent** that
`optimizer.step()` performs.

In [ ]:
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.data = []                       # (reward, action-prob) for the episode
        self.fc1 = nn.Linear(4, 128)         # input: 4-dim CartPole state
        self.fc2 = nn.Linear(128, 2)         # output: 2 action logits
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc2(x), dim=0)  # probabilities over actions

    def put_data(self, item):
        self.data.append(item)

    def train_net(self):
        R = 0
        self.optimizer.zero_grad()
        for r, prob in self.data[::-1]:       # walk the episode backward
            R = r + gamma * R                 # discounted return G_t
            loss = -torch.log(prob) * R       # -log pi(a|s) * G_t
            loss.backward()
        self.optimizer.step()
        self.data = []

## 3. Train

One update per episode. We log the return of every episode so we can plot the learning curve.
The lecture runs 10,000 episodes; **2,000 is enough to watch it clearly learn** (a few minutes
on CPU). Raise `EPISODES` for a stronger agent.

In [ ]:
EPISODES = 2000

env = gym.make('CartPole-v1')
pi = Policy()
returns = []
score = 0.0
print_interval = 20

for n_epi in range(EPISODES):
    s, _ = env.reset()
    s = torch.tensor(s, dtype=torch.float32)
    done = False
    ep_ret = 0.0
    while not done:
        prob = pi(s)                          # action distribution
        m = Categorical(prob)                 # sample -> stochastic policy
        a = m.sample()
        s2, r, term, trunc, _ = env.step(a.item())
        done = term or trunc
        pi.put_data((r, prob[a]))             # store reward and chosen prob
        s = torch.tensor(s2, dtype=torch.float32)
        ep_ret += r
    pi.train_net()                            # one update per episode
    returns.append(ep_ret)
    score += ep_ret
    if n_epi % print_interval == 0 and n_epi != 0:
        print(f'# episode {n_epi:5d}   avg score (last {print_interval}): {score/print_interval:6.1f}')
        score = 0.0
env.close()

## 4. The learning curve

Plot the per-episode return with a 50-episode moving average. Expect a steady climb from ~20
toward a few hundred — and a **noisy** one. REINFORCE is high-variance because the return
$G_t$ is a Monte Carlo estimate; a single unlucky episode sends a large, misleading gradient.

In [ ]:
import numpy as np
r = np.array(returns)
w = 50
ma = np.convolve(r, np.ones(w)/w, mode='valid')

plt.figure(figsize=(8, 4.5))
plt.plot(r, color='#93c5fd', lw=0.8, label='episode return')
plt.plot(np.arange(w-1, len(r)), ma, color='#2563eb', lw=2.2, label=f'{w}-ep moving avg')
plt.axhline(500, color='#16a34a', ls='--', lw=1.4)
plt.xlabel('episode'); plt.ylabel('return')
plt.title('REINFORCE on CartPole-v1')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()

## 5. Watch the trained policy (optional)

Roll out a few greedy episodes (take the most probable action) and print the returns.

In [ ]:
test_env = gym.make('CartPole-v1')
for ep in range(5):
    s, _ = test_env.reset()
    done = False; ret = 0.0
    while not done:
        with torch.no_grad():
            prob = pi(torch.tensor(s, dtype=torch.float32))
        a = torch.argmax(prob).item()         # greedy at test time
        s, r, term, trunc, _ = test_env.step(a)
        done = term or trunc; ret += r
    print(f'test episode {ep}: return {ret:.0f}')
test_env.close()

## Where to go next

- **Cut the variance.** Subtract a **baseline** $b(s)$ from the return: replace $G_t$ with
  $G_t - b(s)$ so you push up only actions that did *better than expected*. A learned baseline
  is a value network — that turns REINFORCE into an **actor-critic** method (Topic 38).
- **Try a harder environment.** Swap `CartPole-v1` for `LunarLander-v3` (discrete). The same
  code works; it just needs a wider network and more episodes.